In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
# load the dataset
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

# Identify numerical and categorical columns
categorical = df.select_dtypes(include=['object', 'str']).columns.tolist()
numerical = df.select_dtypes(exclude=['object', 'str']).columns.tolist()

# The target variable shouldn't be processed as a feature
if 'converted' in numerical:
    numerical.remove('converted')
if 'converted' in categorical:
    categorical.remove('converted')

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

# Q1

In [3]:
mode_industry = df['industry'].mode()[0]
print(f"Most frequent industry: {mode_industry}")

Most frequent industry: technology


# Q2

In [4]:
corr_matrix = df[numerical].corr()

# Check specific pairs
pairs_to_check = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for col1, col2 in pairs_to_check:
    print(f"Correlation between {col1} and {col2}: {corr_matrix.loc[col1, col2]:.4f}")

Correlation between interaction_count and lead_score: 0.9157
Correlation between number_of_courses_viewed and lead_score: 0.7572
Correlation between number_of_courses_viewed and interaction_count: 0.7216
Correlation between annual_income and interaction_count: 0.1228


In [5]:
# Split the dataset 60/20/20
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

# Isolate the target variable
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

# Remove the target variable from the feature sets
del df_train['converted']
del df_val['converted']
del df_test['converted']

# Q3

In [6]:
# Define a function to calculate MI against the target
def calculate_mi(series):
    return mutual_info_score(series, y_train)

# Apply to all categorical columns in the training set
df_mi = df_train[categorical].apply(calculate_mi)
df_mi = df_mi.sort_values(ascending=False).round(2)
print("Mutual Information Scores:")
print(df_mi)

Mutual Information Scores:
lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


# Q4

In [7]:
# One-hot encoding using DictVectorizer
dv = DictVectorizer(sparse=False)

# Convert the DataFrame to a list of dictionaries and fit the DictVectorizer.
# This automatically one-hot encodes the categorical string variables while
# leaving the numerical variables unchanged.
train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

# Train LR model
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Calculate accuracy
y_pred = model.predict(X_val)
acc_original = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {round(acc_original, 2)}")

Validation Accuracy: 0.65


# Q5

In [8]:
features = categorical + numerical
differences = {}

for f in features:
    # Exclude the current feature
    subset = features.copy()
    subset.remove(f)

    # Prepare data without the feature
    train_dict_sub = df_train[subset].to_dict(orient='records')
    val_dict_sub = df_val[subset].to_dict(orient='records')

    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(train_dict_sub)
    X_val_sub = dv_sub.transform(val_dict_sub)

    # Train and evaluate
    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)

    y_pred_sub = model_sub.predict(X_val_sub)
    acc_sub = accuracy_score(y_val, y_pred_sub)

    # Calculate difference (original - new)
    diff = acc_original - acc_sub
    differences[f] = diff

# Display differences
options_to_check = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
for opt in options_to_check:
    print(f"Difference without {opt}: {differences[opt]}")

Difference without lead_source: 0.0030000000000000027
Difference without number_of_courses_viewed: 0.0020000000000000018
Difference without interaction_count: 0.04400000000000004


# Q6

In [9]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]

for c in c_values:
    # Train model with current C
    model_c = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model_c.fit(X_train, y_train)

    # Calculate accuracy
    y_pred_c = model_c.predict(X_val)
    acc_c = accuracy_score(y_val, y_pred_c)

    print(f"C = {c:<8} | Accuracy = {round(acc_c, 3)}")

C = 1e-06    | Accuracy = 0.598
C = 1e-05    | Accuracy = 0.598
C = 0.0001   | Accuracy = 0.613
C = 0.001    | Accuracy = 0.645
